In [1]:
from typing import List, Tuple

def gap_best_fit_repair(cost: List[List[int]],
                        resource: List[List[int]],
                        capacity: List[int]) -> Tuple[List[int], int]:

    m, n = len(cost), len(cost[0])
    assignment = [-1] * n
    remaining = capacity[:]
    total_cost = 0

    # ------------------ STEP 1: BEST FIT ------------------
    jobs = list(range(n))
    jobs.sort(key=lambda j: min(resource[i][j] for i in range(m)), reverse=True)

    for j in jobs:
        best_machine = -1
        min_space_left = float('inf')

        for i in range(m):
            req = resource[i][j]
            if remaining[i] >= req:
                space_left = remaining[i] - req
                if space_left < min_space_left:
                    min_space_left = space_left
                    best_machine = i

        if best_machine != -1:
            assignment[j] = best_machine
            remaining[best_machine] -= resource[best_machine][j]
            total_cost += cost[best_machine][j]

    # ------------------ STEP 2: REPAIR ------------------
    for j in range(n):
        if assignment[j] != -1:
            continue

        for i in range(m):
            req_j = resource[i][j]

            # Try direct placement first
            if remaining[i] >= req_j:
                assignment[j] = i
                remaining[i] -= req_j
                total_cost += cost[i][j]
                break

            # Try swapping with another job k assigned to i
            for k in range(n):
                if assignment[k] == i:
                    for i2 in range(m):
                        if i2 == i:
                            continue

                        if remaining[i2] >= resource[i2][k]:
                            # Move k → i2
                            assignment[k] = i2
                            remaining[i2] -= resource[i2][k]
                            remaining[i] += resource[i][k]
                            total_cost += cost[i2][k] - cost[i][k]

                            # Now try placing j in i
                            if remaining[i] >= req_j:
                                assignment[j] = i
                                remaining[i] -= req_j
                                total_cost += cost[i][j]
                                break
                    if assignment[j] != -1:
                        break
            if assignment[j] != -1:
                break

    return assignment, total_cost


In [2]:
cost = [
[25,18,15,16,18,25,19,18,25,23,19,25,19,21,24,16,18,23,18,17,25,20,24,21,17,19,24,25,17,19,19,25,20,17,21,20,20,17,19,25,25,19,23,24,20,16,21,24,17,15,17,19,15,21,25,16,21,20,24,22],

[21,21,15,23,23,22,20,22,21,24,20,19,23,24,23,23,22,22,15,18,20,19,20,24,22,20,20,20,20,17,20,24,18,19,22,17,21,17,19,17,22,22,18,22,20,17,19,15,15,19,23,22,24,23,17,21,25,23,20,19],

[21,17,24,17,24,15,22,24,20,18,21,23,19,24,23,22,22,21,16,18,21,22,25,22,17,25,22,22,24,22,25,20,16,19,23,20,18,23,23,19,19,21,17,22,15,16,23,24,17,22,23,18,24,16,20,25,19,17,22,21],

[21,20,22,15,23,24,25,22,18,16,23,24,20,21,15,21,17,20,24,22,24,25,16,17,18,20,25,18,24,24,24,22,19,20,15,20,17,19,16,23,15,22,22,24,15,16,20,17,21,24,24,24,24,25,19,24,25,16,22,19],

[24,23,22,19,18,16,23,18,22,24,25,16,23,21,17,18,19,17,25,17,18,23,24,18,24,23,23,15,19,18,24,19,19,24,24,24,15,21,21,24,24,19,23,24,21,23,24,22,18,19,24,15,17,17,25,17,24,16,19,21],

[16,21,19,22,16,16,17,23,25,16,18,22,15,25,20,15,18,23,22,16,17,17,16,24,24,16,17,23,19,23,15,17,24,19,24,24,24,16,22,16,18,23,23,20,16,25,23,17,18,23,24,19,22,19,19,18,21,21,22,17],

[23,23,19,18,19,19,21,24,22,16,17,25,24,17,23,17,20,25,25,22,17,24,17,17,23,23,16,17,19,19,23,24,17,25,18,19,18,16,17,17,24,23,17,19,15,23,22,17,23,21,15,19,18,22,23,16,16,22,16,21],

[24,16,15,25,16,15,20,20,22,18,21,19,21,19,17,17,17,22,25,17,20,25,15,17,20,16,23,20,23,22,23,15,21,24,21,23,19,24,17,24,25,18,24,21,20,18,22,15,16,18,23,24,25,20,20,21,17,15,24,21],

[18,18,18,19,18,24,22,19,25,15,16,23,17,16,18,22,21,18,21,18,20,18,16,24,17,22,24,18,22,18,24,23,23,25,20,23,18,22,21,18,24,17,15,25,23,16,22,24,18,21,16,24,22,18,25,23,23,18,16,22],

[16,18,19,18,18,24,24,23,25,16,20,22,25,16,20,18,19,24,24,25,22,23,23,24,18,24,16,21,17,22,22,16,15,17,19,23,15,19,22,22,23,23,16,20,22,24,20,20,23,16,24,17,22,17,19,25,16,16,19,16]
]


resource = [
[17,25,6,14,16,12,19,19,5,16,23,16,15,22,19,10,25,7,24,9,7,24,22,23,10,25,21,16,13,17,9,6,16,23,11,22,14,22,5,14,9,19,24,16,6,21,19,11,22,22,19,24,12,20,15,17,21,17,25,19],

[6,24,12,24,14,22,11,10,25,12,20,17,18,22,11,15,20,20,21,12,5,5,25,15,24,17,7,21,11,12,24,16,12,5,9,11,14,7,14,9,13,7,25,12,6,10,7,15,13,22,6,10,5,21,5,16,16,13,5,23],

[21,21,7,5,20,16,9,14,22,12,9,8,14,17,22,9,17,8,12,15,24,19,6,8,16,14,14,23,14,23,5,21,17,25,13,23,23,12,20,19,16,15,22,19,18,7,5,20,19,9,23,14,14,11,20,15,25,17,12,13],

[22,14,19,7,5,20,19,5,20,7,12,8,13,18,16,9,23,16,16,9,25,8,13,16,6,8,5,18,12,5,12,16,12,25,5,24,23,23,25,22,17,17,21,24,16,7,17,9,13,24,22,25,25,23,12,12,8,24,14,8],

[17,10,18,13,14,18,6,6,11,9,12,20,15,12,18,18,11,7,9,25,13,9,10,9,15,10,19,15,15,14,25,6,12,21,9,10,18,20,13,17,17,21,15,9,22,11,14,14,13,16,12,5,14,18,24,9,19,14,9,24],

[24,23,6,13,9,17,18,21,10,15,15,18,14,15,11,24,20,24,16,20,23,25,9,21,21,13,9,24,12,8,14,21,16,25,20,23,8,16,20,10,5,16,11,20,10,11,21,15,12,12,21,10,19,18,10,25,23,16,8,25],

[5,21,24,13,21,11,18,12,24,22,20,21,16,18,8,25,7,19,11,12,18,5,15,7,12,5,23,16,12,5,8,24,25,10,13,24,22,20,12,16,18,9,21,17,19,6,21,12,6,9,24,10,8,13,13,13,23,25,8,22],

[9,19,23,7,5,8,16,17,21,15,15,12,21,24,15,6,7,11,5,13,10,16,19,9,22,8,21,17,7,8,25,23,20,8,14,16,5,16,5,10,17,6,18,9,19,9,11,18,23,14,24,13,20,10,22,19,6,15,18,15],

[23,15,16,20,19,23,8,15,24,25,25,5,13,14,12,25,5,11,23,24,18,7,17,10,25,21,13,19,18,6,8,9,21,18,5,12,8,7,19,24,7,9,22,7,9,10,14,24,22,24,15,16,11,19,18,18,16,8,6,8],

[10,14,24,13,18,11,13,24,22,7,12,17,23,12,15,5,5,16,20,22,18,12,20,12,9,5,7,6,25,6,7,13,7,23,21,21,18,8,23,7,11,9,15,19,9,13,22,24,18,14,11,15,15,18,22,13,17,14,11,11]
]


capacity = [79, 67, 74, 73, 67, 78, 73, 68, 73, 69]


assignment, total_cost = gap_best_fit_repair(cost, resource, capacity)
assignment_1_based = [a+1 for a in assignment]

print("Job → Machine assignment:", assignment_1_based)
print("Total cost:", total_cost)


Job → Machine assignment: [6, 10, 8, 6, 4, 2, 8, 1, 6, 2, 4, 1, 10, 10, 2, 10, 1, 5, 0, 4, 1, 0, 8, 5, 3, 0, 0, 3, 5, 6, 1, 7, 5, 0, 0, 4, 0, 9, 0, 2, 0, 3, 10, 3, 7, 8, 0, 9, 7, 9, 7, 0, 0, 4, 0, 9, 6, 2, 0, 5]
Total cost: 938


In [17]:
from typing import List, Tuple
from collections import defaultdict

def gap_best_fit_repair_maximization(cost: List[List[int]],
                                     resource: List[List[int]],
                                     capacity: List[int]) -> Tuple[List[int], int]:
    """
    For MAXIMIZATION version of GAP.
    cost[i][j] = profit of assigning job j to agent i
    We want to maximize total profit while respecting capacity constraints.
    """
    m, n = len(cost), len(cost[0])
    assignment = [-1] * n
    remaining = capacity[:]
    total_profit = 0
    
    # Step 1: Sort job-agent pairs by profit-to-resource ratio (descending)
    # This helps prioritize jobs that give good profit relative to resource consumption
    job_agent_info = []
    for j in range(n):
        for i in range(m):
            if resource[i][j] > 0:
                ratio = cost[i][j] / resource[i][j]
            else:
                ratio = float('inf')  # If no resource needed, prioritize highly
            job_agent_info.append((j, i, cost[i][j], resource[i][j], ratio))
    
    # Sort by ratio (descending) then by profit (descending)
    job_agent_info.sort(key=lambda x: (x[4], x[2]), reverse=True)
    
    # Step 2: Initial assignment - try to assign high-ratio jobs first
    assigned_jobs = set()
    for j, i, profit, req, _ in job_agent_info:
        if j in assigned_jobs:
            continue
        if remaining[i] >= req:
            assignment[j] = i
            remaining[i] -= req
            total_profit += profit
            assigned_jobs.add(j)
    
    # Step 3: For any still unassigned jobs, try more aggressive repair
    for j in range(n):
        if assignment[j] != -1:
            continue
            
        # Try to find the best assignment for this job
        best_agent = -1
        best_profit = -1
        best_remaining_space = -1
        
        for i in range(m):
            req = resource[i][j]
            
            # Try direct placement
            if remaining[i] >= req:
                # Prefer agent with higher profit and less remaining space
                profit_here = cost[i][j]
                if profit_here > best_profit:
                    best_agent = i
                    best_profit = profit_here
                    best_remaining_space = remaining[i] - req
            
            # If no direct placement possible, try to free up space
            else:
                needed_space = req - remaining[i]
                
                # Look for jobs we can move from this agent
                for k in range(n):
                    if assignment[k] == i:
                        job_k_resource = resource[i][k]
                        job_k_profit = cost[i][k]
                        
                        # Check if moving this job would free enough space
                        if job_k_resource >= needed_space:
                            # Try to find another agent for job k
                            for i2 in range(m):
                                if i2 == i:
                                    continue
                                    
                                if remaining[i2] >= resource[i2][k]:
                                    # Calculate net profit change
                                    net_profit_change = (cost[i][j] - job_k_profit + 
                                                         cost[i2][k])
                                    
                                    if net_profit_change > best_profit:
                                        best_agent = i
                                        best_profit = net_profit_change
                                        # Store the swap info
                                        swap_info = (k, i2, job_k_profit, cost[i2][k])
                                        # We'll implement the swap later
                                        break
                        
                        # If we found a good swap, break
                        if best_agent != -1:
                            break
                    # If we found a good swap, break
                    if best_agent != -1:
                        break
        
        # Now implement the best assignment we found
        if best_agent != -1:
            req = resource[best_agent][j]
            
            if remaining[best_agent] >= req:
                # Direct assignment
                assignment[j] = best_agent
                remaining[best_agent] -= req
                total_profit += cost[best_agent][j]
            else:
                # Need to do a swap
                # Find which job to move (we need to re-find this)
                for k in range(n):
                    if assignment[k] == best_agent:
                        if resource[best_agent][k] >= (req - remaining[best_agent]):
                            # Find destination for job k
                            for i2 in range(m):
                                if i2 == best_agent:
                                    continue
                                    
                                if remaining[i2] >= resource[i2][k]:
                                    # Move job k
                                    assignment[k] = i2
                                    remaining[i2] -= resource[i2][k]
                                    remaining[best_agent] += resource[best_agent][k]
                                    total_profit += cost[i2][k] - cost[best_agent][k]
                                    
                                    # Now assign job j
                                    assignment[j] = best_agent
                                    remaining[best_agent] -= req
                                    total_profit += cost[best_agent][j]
                                    break
                            break
    
    # Step 4: Final pass - if any job is still unassigned,
    # assign it to the agent with the most remaining capacity
    # This ensures all jobs are assigned, but may violate capacity
    for j in range(n):
        if assignment[j] == -1:
            # Find agent with maximum remaining capacity
            best_agent = max(range(m), key=lambda i: remaining[i])
            assignment[j] = best_agent
            total_profit += cost[best_agent][j]
            # We'll track this as a potential violation
    
    return assignment, total_profit

In [18]:
def print_assignment_details(assignment: List[int], cost: List[List[int]], 
                           resource: List[List[int]], capacity: List[int]):
    """
    Print detailed assignment information
    """
    m, n = len(cost), len(cost[0])
    
    # Group jobs by machine
    machine_to_jobs = defaultdict(list)
    for job_idx, machine_idx in enumerate(assignment):
        if machine_idx != -1:
            machine_to_jobs[machine_idx].append(job_idx)
    
    print("=" * 80)
    print("DETAILED ASSIGNMENT ANALYSIS")
    print("=" * 80)
    
    # Print job assignments per machine
    print("\nJobs assigned to each machine:")
    print("-" * 80)
    
    total_profit = 0
    capacity_violations = []
    
    for machine in sorted(machine_to_jobs.keys()):
        jobs = machine_to_jobs[machine]
        jobs_sorted = sorted(jobs)
        
        # Calculate resource usage and profit for this machine
        resource_used = 0
        machine_profit = 0
        
        for job in jobs:
            resource_used += resource[machine][job]
            machine_profit += cost[machine][job]
        
        total_profit += machine_profit
        
        print(f"Machine {machine + 1} (Capacity: {capacity[machine]}):")
        print(f"  Jobs: {[j+1 for j in jobs_sorted]}")
        print(f"  Number of jobs: {len(jobs)}")
        print(f"  Resource used: {resource_used} / {capacity[machine]}")
        print(f"  Remaining capacity: {capacity[machine] - resource_used}")
        print(f"  Profit contribution: {machine_profit}")
        
        if resource_used > capacity[machine]:
            capacity_violations.append((machine+1, resource_used, capacity[machine]))
            print(f"  ⚠️ CAPACITY EXCEEDED by {resource_used - capacity[machine]} units!")
        print()
    
    # Print summary
    print("=" * 80)
    print("SUMMARY")
    print("=" * 80)
    
    # Check if all jobs are assigned
    unassigned_jobs = [j+1 for j in range(n) if assignment[j] == -1]
    if unassigned_jobs:
        print(f"✗ {len(unassigned_jobs)} jobs unassigned: {unassigned_jobs}")
    else:
        print("✓ All jobs are assigned")
    
    # Check capacity constraints
    if capacity_violations:
        print("✗ Capacity violations found:")
        for machine, used, cap in capacity_violations:
            print(f"  Machine {machine}: Used {used} > Capacity {cap}")
        print(f"  Total violations: {len(capacity_violations)} machines")
    else:
        print("✓ All capacity constraints satisfied")
    
    print(f"\nTotal profit: {total_profit}")
    
    # Calculate total possible profit (sum of max profit for each job)
    max_possible_profit = 0
    for j in range(n):
        max_profit_for_job = max(cost[i][j] for i in range(m))
        max_possible_profit += max_profit_for_job
    
    print(f"Maximum possible profit (without capacity constraints): {max_possible_profit}")
    print(f"Percentage of maximum achieved: {total_profit/max_possible_profit*100:.2f}%")

In [19]:
cost = [
[25,18,15,16,18,25,19,18,25,23,19,25,19,21,24,16,18,23,18,17,25,20,24,21,17,19,24,25,17,19,19,25,20,17,21,20,20,17,19,25,25,19,23,24,20,16,21,24,17,15,17,19,15,21,25,16,21,20,24,22],

[21,21,15,23,23,22,20,22,21,24,20,19,23,24,23,23,22,22,15,18,20,19,20,24,22,20,20,20,20,17,20,24,18,19,22,17,21,17,19,17,22,22,18,22,20,17,19,15,15,19,23,22,24,23,17,21,25,23,20,19],

[21,17,24,17,24,15,22,24,20,18,21,23,19,24,23,22,22,21,16,18,21,22,25,22,17,25,22,22,24,22,25,20,16,19,23,20,18,23,23,19,19,21,17,22,15,16,23,24,17,22,23,18,24,16,20,25,19,17,22,21],

[21,20,22,15,23,24,25,22,18,16,23,24,20,21,15,21,17,20,24,22,24,25,16,17,18,20,25,18,24,24,24,22,19,20,15,20,17,19,16,23,15,22,22,24,15,16,20,17,21,24,24,24,24,25,19,24,25,16,22,19],

[24,23,22,19,18,16,23,18,22,24,25,16,23,21,17,18,19,17,25,17,18,23,24,18,24,23,23,15,19,18,24,19,19,24,24,24,15,21,21,24,24,19,23,24,21,23,24,22,18,19,24,15,17,17,25,17,24,16,19,21],

[16,21,19,22,16,16,17,23,25,16,18,22,15,25,20,15,18,23,22,16,17,17,16,24,24,16,17,23,19,23,15,17,24,19,24,24,24,16,22,16,18,23,23,20,16,25,23,17,18,23,24,19,22,19,19,18,21,21,22,17],

[23,23,19,18,19,19,21,24,22,16,17,25,24,17,23,17,20,25,25,22,17,24,17,17,23,23,16,17,19,19,23,24,17,25,18,19,18,16,17,17,24,23,17,19,15,23,22,17,23,21,15,19,18,22,23,16,16,22,16,21],

[24,16,15,25,16,15,20,20,22,18,21,19,21,19,17,17,17,22,25,17,20,25,15,17,20,16,23,20,23,22,23,15,21,24,21,23,19,24,17,24,25,18,24,21,20,18,22,15,16,18,23,24,25,20,20,21,17,15,24,21],

[18,18,18,19,18,24,22,19,25,15,16,23,17,16,18,22,21,18,21,18,20,18,16,24,17,22,24,18,22,18,24,23,23,25,20,23,18,22,21,18,24,17,15,25,23,16,22,24,18,21,16,24,22,18,25,23,23,18,16,22],

[16,18,19,18,18,24,24,23,25,16,20,22,25,16,20,18,19,24,24,25,22,23,23,24,18,24,16,21,17,22,22,16,15,17,19,23,15,19,22,22,23,23,16,20,22,24,20,20,23,16,24,17,22,17,19,25,16,16,19,16]
]


resource = [
[17,25,6,14,16,12,19,19,5,16,23,16,15,22,19,10,25,7,24,9,7,24,22,23,10,25,21,16,13,17,9,6,16,23,11,22,14,22,5,14,9,19,24,16,6,21,19,11,22,22,19,24,12,20,15,17,21,17,25,19],

[6,24,12,24,14,22,11,10,25,12,20,17,18,22,11,15,20,20,21,12,5,5,25,15,24,17,7,21,11,12,24,16,12,5,9,11,14,7,14,9,13,7,25,12,6,10,7,15,13,22,6,10,5,21,5,16,16,13,5,23],

[21,21,7,5,20,16,9,14,22,12,9,8,14,17,22,9,17,8,12,15,24,19,6,8,16,14,14,23,14,23,5,21,17,25,13,23,23,12,20,19,16,15,22,19,18,7,5,20,19,9,23,14,14,11,20,15,25,17,12,13],

[22,14,19,7,5,20,19,5,20,7,12,8,13,18,16,9,23,16,16,9,25,8,13,16,6,8,5,18,12,5,12,16,12,25,5,24,23,23,25,22,17,17,21,24,16,7,17,9,13,24,22,25,25,23,12,12,8,24,14,8],

[17,10,18,13,14,18,6,6,11,9,12,20,15,12,18,18,11,7,9,25,13,9,10,9,15,10,19,15,15,14,25,6,12,21,9,10,18,20,13,17,17,21,15,9,22,11,14,14,13,16,12,5,14,18,24,9,19,14,9,24],

[24,23,6,13,9,17,18,21,10,15,15,18,14,15,11,24,20,24,16,20,23,25,9,21,21,13,9,24,12,8,14,21,16,25,20,23,8,16,20,10,5,16,11,20,10,11,21,15,12,12,21,10,19,18,10,25,23,16,8,25],

[5,21,24,13,21,11,18,12,24,22,20,21,16,18,8,25,7,19,11,12,18,5,15,7,12,5,23,16,12,5,8,24,25,10,13,24,22,20,12,16,18,9,21,17,19,6,21,12,6,9,24,10,8,13,13,13,23,25,8,22],

[9,19,23,7,5,8,16,17,21,15,15,12,21,24,15,6,7,11,5,13,10,16,19,9,22,8,21,17,7,8,25,23,20,8,14,16,5,16,5,10,17,6,18,9,19,9,11,18,23,14,24,13,20,10,22,19,6,15,18,15],

[23,15,16,20,19,23,8,15,24,25,25,5,13,14,12,25,5,11,23,24,18,7,17,10,25,21,13,19,18,6,8,9,21,18,5,12,8,7,19,24,7,9,22,7,9,10,14,24,22,24,15,16,11,19,18,18,16,8,6,8],

[10,14,24,13,18,11,13,24,22,7,12,17,23,12,15,5,5,16,20,22,18,12,20,12,9,5,7,6,25,6,7,13,7,23,21,21,18,8,23,7,11,9,15,19,9,13,22,24,18,14,11,15,15,18,22,13,17,14,11,11]
]

capacity = [79, 67, 74, 73, 67, 78, 73, 68, 73, 69]

assignment, total_profit = gap_best_fit_repair_maximization(cost, resource, capacity)
print_assignment_details(assignment, cost, resource, capacity)

DETAILED ASSIGNMENT ANALYSIS

Jobs assigned to each machine:
--------------------------------------------------------------------------------
Machine 1 (Capacity: 79):
  Jobs: [9, 18, 32, 39, 45, 48]
  Number of jobs: 6
  Resource used: 40 / 79
  Remaining capacity: 39
  Profit contribution: 136

Machine 2 (Capacity: 67):
  Jobs: [21, 34, 42, 51, 53, 55, 59]
  Number of jobs: 7
  Resource used: 38 / 67
  Remaining capacity: 29
  Profit contribution: 145

Machine 3 (Capacity: 74):
  Jobs: [3, 11, 23, 24, 31, 47, 50]
  Number of jobs: 7
  Resource used: 49 / 74
  Remaining capacity: 25
  Profit contribution: 162

Machine 4 (Capacity: 73):
  Jobs: [5, 8, 13, 20, 25, 27, 30, 56, 57]
  Number of jobs: 9
  Resource used: 68 / 73
  Remaining capacity: 5
  Profit contribution: 203

Machine 5 (Capacity: 67):
  Jobs: [2, 7, 10, 14, 36, 52]
  Number of jobs: 6
  Resource used: 52 / 67
  Remaining capacity: 15
  Profit contribution: 130

Machine 6 (Capacity: 78):
  Jobs: [41, 43]
  Number of jobs: